In [1]:
import numpy as np
from scipy.odr import *

The following function performs a linear fit of two equal-lengthed sets of data.

The returned values are the fit parameters (intercept and slope), the R^2 of the fit, and the covariance parameters of the fit.

In [2]:
def linear_fit_1var (xdata, ydata):
    x = xdata
    y = ydata
    
    data = pandas.DataFrame({'x': x, 'y': y})
    model = ols("y ~ x", data).fit()
    # Parameters are returned in the format intercept, slope
    return model._results.params, model._results.rsquared, model.bse, model.normalized_cov_params

The following function is the primary function of this document. Using a set of stellar data (star_data) with data on the luminosity, mass, and mass-loss rates of the stars, it will iterate between two specified values with a specified step size to determine the exponents of a mass-loss rate formula.

Data on the stars should be specified column-wise in solar units and years. The location of the luminosity, mass, and mass-loss rate data should be specified outside the function, and the locations of the columns in the data sets we worked with has been included as an example. These are identified with the variables 'LIndex', 'MIndex', and 'MdotIndex' respectively.

The function is written to iterate between two values, intial_n and terminal_n. By default, these are 0.5 and 3.0 but may be adjusted with a keyword argument. The step size is specified by the keyword delta_n, by default 0.1, and may be adjusted as well.

The percentage of stars used to define the mean is specified with the keyword percentile_to_trim. By default this is 0.05, meaning the 5% of stars furthest from the mean mass-loss rate are not included when measuring the range. It may be changed to any value between 0 and 1, but user judgment should be used to determine what is reasonable.

After completing the calculations for all n between the initial and terminal values, it will print the value of n that gives a slope of a Calculated vs. Observed mass-loss rate fit closest to 1. The associated LM strip values (strip fit line parameters, strip height, strip width, exponents, fit parameters, and fit quality statistics) are then printed. The function then returns the exponents of the fit, called A, B, and C here which corespond to logA, B, and C in the article.

In [4]:
MdotIndex=5 # Column of imported files that has Mdot data
MdotErrorIndex=6 # Absolute error in Mdot
MdotDustIndex=2 # Column of imported files that has Mdot_dust Data
LIndex=8 # Column of imported files that has LBol data
LErrorIndex=9 # Absolute error in L
PIndex=13 # Column of imported files that has Period data
JIndex=14 # Column of imported files that has J-band magnitude
HIndex=15 # Column of imported files that has J-band magnitude
KIndex=16 # Column of imported files that has J-band magnitude
ThreePointSixIndex=17 # Column of imported files that has 3.6 band magnitude
FourPointFiveIndex=18 # Column of imported files that has 3.6 band magnitude
FivePointEightIndex=19 # Column of imported files that has 3.6 band magnitude
EightPointZeroIndex=20 # Column of imported files that has 8.0 band magnitude
MIndex=22 # Column of imported files that has Mass data
RIndex=23 # Column of imported files that has Radius data
TeffObsvIndex=12 # Column of imported files that has Observed Teff Data
TeffCalcIndex=24 # Column of imported files that has Calculated Teff Data
tauIndex=27 # Column of imported files that has Optical Depth data

In [5]:
def LM_Strip_Find_Exponents(star_data, initial_n=0.5, terminal_n=3.0, delta_n=0.1,percentile_to_trim = 0.05):
    '''
    Goal: script that calculates A, B, C for fit. Then, take those values and compare them to ideal. Knowing that,
    compare to previous fit. If better, continue. If worse, increase resolution then continue from previous.
    Repeat until a limit on the number of iterations is reached or we are as close to 1 as we can measure with
    numerical precision.
    '''
    '''
    First, we will handle all of the calculations that are simply properties of the data set. No need to recalculate
    these on every iteration. These calculations are finding the line of best fit for the data (presumably the deathline),
    finding the standard deviation of the data around the deathline, calculating the vertical displacement of the upper and 
    lower bounds given a multiplier n, calculating the horizontal displacement of the upper and lower bounds from the fit line,
    using those two calculations to determine the height and width of the strip (respectively), and sorting out the data found 
    inside our LM strip.
    '''
    logLData = np.log10(star_data[LIndex])
    logMData = np.log10(star_data[MIndex])
    logMdotData = np.log10(star_data[MdotIndex])     
    deathline_fit = linear_fit_1var(logLData,logMData)
    deathline_coeffs = deathline_fit[0]
    print('deathline_fit:',deathline_fit)
    line_x_vals = np.asarray([np.nanmin(np.log10(star_data[LIndex])),np.nanmax(np.log10(star_data[LIndex]))])
    line_y_vals = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals
    stnd_dev_around_DL = np.nanstd(logMData-deathline_coeffs[0]-deathline_coeffs[1]*logLData)
                                   
    i = 0
    n = initial_n
    
    n_array = []
    Exponents_array = []
    fit_qual_array = []
    stand_dev_array = []
    rsq_array = []
    deltas_array = []
    while n<terminal_n:
        n_array.append(n)
        bound_size = n*stnd_dev_around_DL*np.cos(np.arctan(deathline_coeffs[1]))
        line_y_vals_up = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals + bound_size
        line_y_vals_down = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals - bound_size
        # Find width of deltaLogL and deltaLogM using fit line, and plot them
        logL_left = (np.average(line_y_vals) - deathline_coeffs[0]-bound_size)/deathline_coeffs[1]
        logL_right = (np.average(line_y_vals) - deathline_coeffs[0]+bound_size)/deathline_coeffs[1]
        delta_logL = logL_right - logL_left
        delta_logM = line_y_vals_up[1] - line_y_vals_down[1]
        star_data_inside_bound = np.asarray([])
        for l in range(0, len(star_data[MIndex])):
            if logMData[l] <= deathline_coeffs[0] + deathline_coeffs[1]*logLData[l] + bound_size:
                if logMData[l] >= deathline_coeffs[0] + deathline_coeffs[1]*logLData[l] - bound_size:
                    if len(star_data_inside_bound) == 0:
                        star_data_inside_bound = np.transpose(star_data)[l]
                    else:
                        star_data_inside_bound = np.vstack((star_data_inside_bound,np.transpose(star_data)[l]))
        star_data_inside_bound = np.transpose(star_data_inside_bound)
        logMdotBoundedData = np.log10(star_data_inside_bound[MdotIndex])
        logMdotBoundedData_mean = np.nanmean(logMdotBoundedData)
        logMdotBoundedData_std = np.nanstd(logMdotBoundedData)
        logMBoundedData = np.log10(star_data_inside_bound[MIndex])
        logLBoundedData = np.log10(star_data_inside_bound[LIndex])

        '''
        Now onto the loop. For some number of iterations, we are going to seek better fits for logMdot = A + BlogL + ClogM.
        From previous experimentation (see above function, fit_golf), we know that as we adjust m/n, the significance of the fit
        is more-or-less fixed using this method. Therefore, we can keep iterating to improve the quality of the fit. We are doing
        this over n and are adjusting m to be to trim the most extreme mass-loss rates
        '''
        logMdot_spread_from_mean = np.abs(logMdotBoundedData - logMdotBoundedData_mean)
        num_to_check = int(np.round_(percentile_to_trim*len(logMdot_spread_from_mean)))
        #print('num_to_check:',num_to_check)
        indices_to_trim = logMdot_spread_from_mean.argsort()[-num_to_check:]
        trimmed_logMdotBoundedData = np.delete(logMdotBoundedData,indices_to_trim)
        delta_logMdot = np.nanmax(trimmed_logMdotBoundedData)-np.nanmin(trimmed_logMdotBoundedData)
        deltas_array.append([delta_logL, delta_logM,delta_logMdot])
        # Calculate B = deltaLogMdot/deltaLogL and C = deltaLogMdot/deltaLogM
        B = delta_logMdot/delta_logL
        C = -delta_logMdot/delta_logM
        '''
        We can force A to be whatever is necessary for the mean residual to be 0
        <logMdot_obsv - logMdot_calc> = 0 = sum(logMdot_obsv[i] - (A + BlogL[i] + ClogM[i]))/len(data)  
        => <A> = sum(logMdot_obsv[i] - BlogL[i] - ClogM[i])/len(data)
        '''
        A = 0
        for k in range(0, np.shape(star_data)[1]):
            if np.isnan(logMdotData[k])==True:
                #print('Index ',i,' skipped due to NaN in logMdot')
                continue
            elif np.isnan(logLData[k])==True:
                #print('Index ',i,' skipped due to NaN in logL')
                continue
            elif np.isnan(logMData[k])==True:
                #print('Index ',i,' skipped due to NaN in logM')
                continue
            else:
                A = A + logMdotData[k]-B*logLData[k] - C*logMData[k]
        A = A/np.shape(star_data)[1]
        logMdot_predict = A + B*np.log10(star_data[LIndex]) + C*np.log10(star_data[MIndex])
        correction = np.nanmean(np.log10(star_data[MdotIndex])-logMdot_predict)
        A = A + correction
        calc_logMdot = A + B*logLData + C*logMData
        Exponents_array.append([A,B,C])
        fit = linear_fit_1var(logMdotData, calc_logMdot)
        fit_params = fit[0]
        fitRSq = fit[1]
        fit_qual_array.append(fit_params[1])
        rsq_array.append(fitRSq)
        residuals = logMdotData-calc_logMdot
        stand_dev_array.append( np.nanstd(residuals))
    
        n = n + delta_n
    fit_qual_array = np.asarray(fit_qual_array)
    fit_qual_testing = np.abs(fit_qual_array - 1)
    where_min = np.argmin(fit_qual_testing)
    
    print('Optimal fit found at ',where_min)
    print('n=',n_array[where_min])
    print('Fit quality: Slope=',fit_qual_array[where_min])
    print('Fit significance: R^2=', rsq_array[where_min])
    print('Fit standard dev: sigma=', stand_dev_array[where_min])
    print('delta logL, delta logM, delta logMdot:', deltas_array[where_min])
    print('log A, B, C:', Exponents_array[where_min])
    print('\n')
    
    return A, B, C

